With the Gold Pipeline we now can do analysis.  The goal here is to find drug and reaction pairs that happen more than what is predicted, which is how we can spot possible safety problems.  First, we build the pairs through the Silve tables and count them into a contingency table.  Then we run disproportionality math, PRR, ROR, and a chi square test to analyze how unexpected each pair is.  Then we flag the ones that are above a certain threshhold and write them out for reporting.

In [ ]:
%pip install pyspark==3.5.0 delta-spark==3.2.0 -q

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
PROJECT_ROOT = "/content/drive/MyDrive/faers-data-pipeline"
sys.path.insert(0, PROJECT_ROOT)

from src.utils.spark_session import get_spark
spark = get_spark("FAERS-Gold")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

SILVER_DIR = os.path.join(PROJECT_ROOT, "data/silver")
GOLD_DIR = os.path.join(PROJECT_ROOT, "data/gold")
os.makedirs(GOLD_DIR, exist_ok=True)

print(f"✓ Spark {spark.version} ready")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 25.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.
Mounted at /content/drive
✓ Spark 3.5.0 ready


We only care about the primary drug, so we can start by filteirng to those and dropping the rest.  Then we join them to the reactions on the report id.

In [ ]:
from pyspark.sql import functions as F

drug_df = spark.read.format("delta").load(os.path.join(SILVER_DIR, "drug"))
reac_df = spark.read.format("delta").load(os.path.join(SILVER_DIR, "reac"))

# Filter to Primary Suspect drugs only
suspect_drugs = drug_df.filter(F.col("role_cod") == "PS") \
    .select("primaryid", "drugname")

print(f"Total drug records:    {drug_df.count():,}")
print(f"Primary Suspect only:  {suspect_drugs.count():,}")
print(f"Reaction records:      {reac_df.count():,}")

# Join to get one row per (primaryid, drugname, reaction) triple
# Use distinct to avoid counting the same report twice for the same pair
drug_event_pairs = suspect_drugs.join(
    reac_df.select("primaryid", "pt"),
    on="primaryid"
).select("primaryid", "drugname", "pt").distinct()

pair_count = drug_event_pairs.count()
unique_drugs = drug_event_pairs.select("drugname").distinct().count()
unique_events = drug_event_pairs.select("pt").distinct().count()
N = drug_event_pairs.select("primaryid").distinct().count()

print(f"\nDrug-event pairs:      {pair_count:,}")
print(f"Unique drugs:          {unique_drugs:,}")
print(f"Unique reactions:      {unique_events:,}")
print(f"Unique reports (N):    {N:,}")

Total drug records:    3,519,718
Primary Suspect only:  768,878
Reaction records:      2,683,365

Drug-event pairs:      2,644,909
Unique drugs:          5,609
Unique reactions:      14,473
Unique reports (N):    767,774


To measure disproportionality, we need to focus on 4 things.  How many reports name both the frug and the reaction? How many name the drug but not the reaction? How many name the reaction but not the drug? How many name neither?  Here we will find those 4 counts.  

In [ ]:
# a = reports with both this drug AND this event
drug_event_counts = drug_event_pairs.groupBy("drugname", "pt") \
    .agg(F.countDistinct("primaryid").alias("a"))

# drug_total = all reports mentioning this drug (a + b)
drug_totals = drug_event_pairs.groupBy("drugname") \
    .agg(F.countDistinct("primaryid").alias("drug_total"))

# event_total = all reports mentioning this event (a + c)
event_totals = drug_event_pairs.groupBy("pt") \
    .agg(F.countDistinct("primaryid").alias("event_total"))

# Assemble the full contingency table
contingency = (drug_event_counts
    .join(drug_totals, on="drugname")
    .join(event_totals, on="pt")
    .withColumn("b", F.col("drug_total") - F.col("a"))
    .withColumn("c", F.col("event_total") - F.col("a"))
    .withColumn("d", F.lit(N) - F.col("a") - F.col("b") - F.col("c"))
    .withColumn("N", F.lit(N))
)

# Cast to double to prevent integer overflow in multiplication
for c in ["a", "b", "c", "d", "N"]:
    contingency = contingency.withColumn(c, F.col(c).cast("double"))

print(f"Contingency table rows (drug-event pairs): {contingency.count():,}")

Contingency table rows (drug-event pairs): 586,375


PRR and ROR both measure how much more often the reaction shows up with this drug than without it. The chi square test tests whether that is just noise or not. A pair counts as a signal only when it clears Evans' criteria, which asks for a strong ratio, a solid chi square, and at least a handful of real reports behind it.

In [ ]:
# Continuity correction. Add 0.5 when any cell is zero.
contingency = (contingency
    .withColumn("_needs_correction",
        (F.col("a") == 0) | (F.col("b") == 0) |
        (F.col("c") == 0) | (F.col("d") == 0))
    .withColumn("a_adj",
        F.when(F.col("_needs_correction"), F.col("a") + 0.5).otherwise(F.col("a")))
    .withColumn("b_adj",
        F.when(F.col("_needs_correction"), F.col("b") + 0.5).otherwise(F.col("b")))
    .withColumn("c_adj",
        F.when(F.col("_needs_correction"), F.col("c") + 0.5).otherwise(F.col("c")))
    .withColumn("d_adj",
        F.when(F.col("_needs_correction"), F.col("d") + 0.5).otherwise(F.col("d")))
)

# PRR = (a/(a+b)) / (c/(c+d))
contingency = contingency.withColumn("PRR",
    (F.col("a_adj") / (F.col("a_adj") + F.col("b_adj"))) /
    (F.col("c_adj") / (F.col("c_adj") + F.col("d_adj")))
)

# ROR = (a*d) / (b*c)
contingency = contingency.withColumn("ROR",
    (F.col("a_adj") * F.col("d_adj")) / (F.col("b_adj") * F.col("c_adj"))
)

# Chi square with Yates' correction
# χ² = N * (|ad - bc| - N/2)² / ((a+b)(c+d)(a+c)(b+d))
contingency = contingency.withColumn("chi2",
    (F.col("N") *
     F.pow(
         F.abs(F.col("a") * F.col("d") - F.col("b") * F.col("c")) - F.col("N") / 2,
         2
     )) /
    ((F.col("a") + F.col("b")) *
     (F.col("c") + F.col("d")) *
     (F.col("a") + F.col("c")) *
     (F.col("b") + F.col("d")))
)

# 95% CI for ROR uses SE(ln(ROR)) = sqrt(1/a + 1/b + 1/c + 1/d)
contingency = (contingency
    .withColumn("SE_ln_ROR",
        F.sqrt(
            1 / F.col("a_adj") + 1 / F.col("b_adj") +
            1 / F.col("c_adj") + 1 / F.col("d_adj")
        ))
    .withColumn("ROR_lower",
        F.exp(F.log(F.col("ROR")) - 1.96 * F.col("SE_ln_ROR")))
    .withColumn("ROR_upper",
        F.exp(F.log(F.col("ROR")) + 1.96 * F.col("SE_ln_ROR")))
)

# Apply Evans' criteria. PRR >= 2, chi2 >= 4, a >= 3.
contingency = contingency.withColumn("is_signal",
    (F.col("PRR") >= 2) & (F.col("chi2") >= 4) & (F.col("a") >= 3)
)

total_pairs = contingency.count()
signal_count = contingency.filter(F.col("is_signal")).count()

print(f"Total drug-event pairs evaluated: {total_pairs:,}")
print(f"Signals flagged (Evans' criteria): {signal_count:,}")
print(f"Signal rate: {signal_count / total_pairs * 100:.2f}%")

Total drug-event pairs evaluated: 586,375
Signals flagged (Evans' criteria): 85,230
Signal rate: 14.54%


Let's see the top thirty pairs by ROR.  These pairs are thwere the reaction is most disproportionately tied to the drug.

In [ ]:
signals_df = contingency.filter(F.col("is_signal"))

print("Top 30 safety signals (by ROR):\n")
(signals_df
    .select(
        "drugname", "pt",
        F.col("a").cast("int").alias("report_count"),
        F.round("PRR", 2).alias("PRR"),
        F.round("ROR", 2).alias("ROR"),
        F.round("ROR_lower", 2).alias("ROR_95CI_lower"),
        F.round("ROR_upper", 2).alias("ROR_95CI_upper"),
        F.round("chi2", 2).alias("chi2"),
    )
    .orderBy(F.col("ROR").desc())
    .limit(30)
    .toPandas()
)

Top 30 safety signals (by ROR):



,drugname,pt,report_count,PRR,ROR,ROR_95CI_lower,ROR_95CI_upper,chi2
0,GELATIN,POST EMBOLISATION SYNDROME,4,987131.57,2763966.60,115646.60,66059107.91,391882.75
1,MENINGOCOCCAL GROUP B VACCINE,MENINGOCOCCAL INFECTION,19,42774.98,1710960.09,99365.75,29460699.34,384158.14
2,PROSTIN E2,UTERINE HYPERSTIMULATION,3,287913.75,1151652.00,81052.67,16363461.98,239927.19
3,COLLAGENASE CLOSTRIDIUM HISTOLYTICUM,FRACTURE OF PENIS,11,44592.35,1070193.48,60539.69,18918399.29,285000.12
4,ELMIRON,PIGMENTARY MACULOPATHY,153,476833.15,691792.72,42986.45,11133208.75,236614.04
5,PARAGARD T 380A,FOREIGN BODY IN REPRODUCTIVE TRACT,791,422933.03,584531.25,36515.65,9356995.29,211356.07
6,HYDRALAZINE HCL,IGM NEPHROPATHY,6,415865.67,570329.69,30932.65,10515617.63,168293.64
7,AVOBENZONE\HOMOSALATE\OCTISALATE\OCTOCRYLENE O...,PRODUCT CONTAMINATION CHEMICAL,4,55279.51,552786.12,28248.70,10817222.47,146953.57
8,CRANGON SHRIMP,FALSE NEGATIVE INVESTIGATION RESULT,4,55279.51,552786.12,28248.70,10817222.47,146953.57
9,NETSPOT,SCAN GALLIUM ABNORMAL,3,383881.00,511841.00,24870.13,10533969.62,123038.27


Here we check a handful of common drug and reaction pairs and see if hte pipeline can recognize them.

In [ ]:
known_signals = [
    ("METFORMIN", "LACTIC ACIDOSIS"),
    ("WARFARIN", "HAEMORRHAGE"),
    ("METHOTREXATE", "PANCYTOPENIA"),
    ("ISOTRETINOIN", "DEPRESSION"),
    ("INFLIXIMAB", "TUBERCULOSIS"),
    ("SIMVASTATIN", "RHABDOMYOLYSIS"),
    ("ATORVASTATIN", "RHABDOMYOLYSIS"),
]

print("Known signal validation:\n")
print(f"{'Drug':<20} {'Event':<25} {'N':>5} {'PRR':>8} {'ROR':>8} {'χ²':>8} {'Signal?'}")
print("-" * 82)

for drug, event in known_signals:
    row = contingency.filter(
        (F.upper(F.col("drugname")) == drug) &
        (F.upper(F.col("pt")) == event)
    ).first()

    if row:
        flag = "✓ YES" if row.is_signal else "✗ NO"
        print(f"{drug:<20} {event:<25} {int(row.a):>5} {row.PRR:>8.2f} "
              f"{row.ROR:>8.2f} {row.chi2:>8.2f} {flag}")
    else:
        print(f"{drug:<20} {event:<25}   — not found in data")

Known signal validation:

Drug                 Event                         N      PRR      ROR       χ² Signal?
----------------------------------------------------------------------------------
METFORMIN            LACTIC ACIDOSIS             574   269.57   376.33 89800.25 ✓ YES
WARFARIN             HAEMORRHAGE                   5     8.90     9.25    27.72 ✓ YES
METHOTREXATE         PANCYTOPENIA                235    19.96    21.01  3779.96 ✓ YES
ISOTRETINOIN         DEPRESSION                   26    10.83    11.83   223.39 ✓ YES
INFLIXIMAB           TUBERCULOSIS                 18    20.24    20.51   300.28 ✓ YES
SIMVASTATIN          RHABDOMYOLYSIS               38    99.73   119.17  3516.74 ✓ YES
ATORVASTATIN         RHABDOMYOLYSIS              130    43.40    46.42  4817.99 ✓ YES



This counts how many signals each drug has and shows the drugs with the most signals. A long list of flags for one drug can point to a risky product or to a drug that simply appears in a huge number of reports.

In [ ]:
signals_per_drug = (signals_df
    .groupBy("drugname")
    .agg(
        F.count("*").alias("num_signals"),
        F.round(F.max("ROR"), 2).alias("max_ROR"),
        F.round(F.max("PRR"), 2).alias("max_PRR"),
    )
    .orderBy(F.col("num_signals").desc())
)

print(f"Drugs with at least one signal: {signals_per_drug.count():,}\n")

print("Top 20 drugs by number of flagged signals:")
signals_per_drug.limit(20).toPandas()

Drugs with at least one signal: 2,882

Top 20 drugs by number of flagged signals:


,drugname,num_signals,max_ROR,max_PRR
0,HUMAN IMMUNOGLOBULIN G,908,876.12,873.12
1,INFLECTRA,776,1234.92,1233.92
2,METHOTREXATE,774,1523.80,1520.77
3,RITUXIMAB,767,817.14,816.27
4,ADALIMUMAB,691,1954.82,1952.33
5,HUMIRA,594,223.48,223.42
6,OCREVUS,537,706.31,705.98
7,MYCOPHENOLATE MOFETIL,531,2507.34,2503.24
8,COSENTYX,481,629.21,627.14
9,RINVOQ,429,535.14,534.39



This cell finds the reports that ended in death, puts them into flagged paris and ranks the signals by how many deaths are tied to them.  

In [ ]:
outc_df = spark.read.format("delta").load(os.path.join(SILVER_DIR, "outc"))
demo_df = spark.read.format("delta").load(os.path.join(SILVER_DIR, "demo"))

# Get primaryids with death outcome
death_ids = outc_df.filter(F.col("outc_cod") == "DE").select("primaryid").distinct()

# Get suspect drug and reaction pairs that led to death
suspect_drugs = drug_df.filter(F.col("role_cod") == "PS").select("primaryid", "drugname")
death_drug_events = (suspect_drugs
    .join(death_ids, on="primaryid")
    .join(reac_df.select("primaryid", "pt"), on="primaryid")
    .groupBy("drugname", "pt")
    .agg(F.countDistinct("primaryid").alias("death_reports"))
)

# Join with signals
signals_with_death = (signals_df
    .join(death_drug_events, on=["drugname", "pt"], how="left")
    .withColumn("death_reports", F.coalesce(F.col("death_reports"), F.lit(0)))
    .filter(F.col("death_reports") > 0)
)

print(f"Signals associated with death outcomes: {signals_with_death.count():,}\n")

print("Top 20 signals with highest death report counts:")
(signals_with_death
    .select(
        "drugname", "pt",
        F.col("a").cast("int").alias("total_reports"),
        "death_reports",
        F.round("PRR", 2).alias("PRR"),
        F.round("ROR", 2).alias("ROR"),
    )
    .orderBy(F.col("death_reports").desc())
    .limit(20)
    .toPandas()
)

Signals associated with death outcomes: 25,469

Top 20 signals with highest death report counts:


,drugname,pt,total_reports,death_reports,PRR,ROR
0,ELIQUIS,DEATH,1033,1032,5.20,6.19
1,FARXIGA,DEATH,852,850,11.86,20.19
2,NUPLAZID,DEATH,728,727,5.06,6.00
3,VENCLEXTA,DEATH,616,616,6.62,8.44
4,TAGRISSO,DEATH,605,605,9.91,15.06
5,FENTANYL,DRUG ABUSE,584,551,207.70,487.12
6,IMBRUVICA,DEATH,395,394,4.55,5.27
7,ACETAMINOPHEN,TOXICITY TO VARIOUS AGENTS,955,366,38.84,54.16
8,CARBIDOPA\LEVODOPA,DEATH,349,349,4.26,4.88
9,VYNDAMAX,DEATH,339,337,5.88,7.24



This saves two tables. The full contingency table keeps every pair and its scores. The signals table keeps only the flagged pairs, so anyone who wants the headline results can read them.  

In [ ]:
from pyspark.sql.functions import current_timestamp, lit

# 1. Full contingency table (all pairs, for future analysis)
contingency_output = (contingency
    .select(
        "drugname", "pt",
        F.col("a").cast("int").alias("report_count"),
        F.col("b").cast("int").alias("drug_no_event"),
        F.col("c").cast("int").alias("event_no_drug"),
        F.col("d").cast("int").alias("neither"),
        F.round("PRR", 4).alias("PRR"),
        F.round("ROR", 4).alias("ROR"),
        F.round("ROR_lower", 4).alias("ROR_95CI_lower"),
        F.round("ROR_upper", 4).alias("ROR_95CI_upper"),
        F.round("chi2", 4).alias("chi2"),
        "is_signal",
    )
    .withColumn("_computed_ts", current_timestamp())
    .withColumn("_run_label", lit("2024_Q1Q2"))
)

contingency_path = os.path.join(GOLD_DIR, "drug_event_contingency")
(contingency_output.write
    .format("delta")
    .mode("overwrite")
    .save(contingency_path))

print(f"✓ Full contingency table: {contingency_output.count():,} pairs")
print(f"  → {contingency_path}")

# 2. Flagged signals only (filtered view for quick access)
signals_output = contingency_output.filter(F.col("is_signal"))

signals_path = os.path.join(GOLD_DIR, "safety_signals")
(signals_output.write
    .format("delta")
    .mode("overwrite")
    .save(signals_path))

print(f"✓ Safety signals table:   {signals_output.count():,} signals")
print(f"  → {signals_path}")

✓ Full contingency table: 586,375 pairs
  → /content/drive/MyDrive/faers-data-pipeline/data/gold/drug_event_contingency
✓ Safety signals table:   85,230 signals
  → /content/drive/MyDrive/faers-data-pipeline/data/gold/safety_signals


This reads the tables back and confirms their counts, prints the signals schema, and show sa sample of the strongest signals.  

In [ ]:
contingency_gold = spark.read.format("delta").load(
    os.path.join(GOLD_DIR, "drug_event_contingency"))
signals_gold = spark.read.format("delta").load(
    os.path.join(GOLD_DIR, "safety_signals"))

print("Gold Layer Summary:")
print(f"  Drug-event contingency table: {contingency_gold.count():,} pairs")
print(f"  Flagged safety signals:       {signals_gold.count():,} signals")
print(f"  Signal rate:                  "
      f"{signals_gold.count() / contingency_gold.count() * 100:.2f}%")

# Schema check
print("\nSignals table schema:")
signals_gold.printSchema()

# Quick sample
print("\nTop 10 signals by report count:")
(signals_gold
    .orderBy(F.col("report_count").desc())
    .select("drugname", "pt", "report_count", "PRR", "ROR",
            "ROR_95CI_lower", "ROR_95CI_upper", "chi2")
    .limit(10)
    .toPandas()
)

Gold Layer Summary:
  Drug-event contingency table: 586,375 pairs
  Flagged safety signals:       85,230 signals
  Signal rate:                  14.54%

Signals table schema:
root
 |-- drugname: string (nullable = true)
 |-- pt: string (nullable = true)
 |-- report_count: integer (nullable = true)
 |-- drug_no_event: integer (nullable = true)
 |-- event_no_drug: integer (nullable = true)
 |-- neither: integer (nullable = true)
 |-- PRR: double (nullable = true)
 |-- ROR: double (nullable = true)
 |-- ROR_95CI_lower: double (nullable = true)
 |-- ROR_95CI_upper: double (nullable = true)
 |-- chi2: double (nullable = true)
 |-- is_signal: boolean (nullable = true)
 |-- _computed_ts: timestamp (nullable = true)
 |-- _run_label: string (nullable = true)


Top 10 signals by report count:


,drugname,pt,report_count,PRR,ROR,ROR_95CI_lower,ROR_95CI_upper,chi2
0,REPATHA,DRUG DOSE OMISSION BY DEVICE,9741,57.0189,123.2261,118.6927,127.9326,231502.1846
1,REPATHA,DEVICE DIFFICULT TO USE,7973,123.1221,220.3928,210.0867,231.2045,247727.9126
2,DUPIXENT,PRURITUS,7259,9.5044,11.1664,10.8280,11.5153,35798.1173
3,DUPIXENT,PRODUCT USE IN UNAPPROVED INDICATION,4204,6.6260,7.2143,6.9506,7.4880,14546.8626
4,MOUNJARO,INCORRECT DOSE ADMINISTERED,4203,21.3654,28.6033,27.4612,29.7929,57036.8623
5,DUPIXENT,DRY SKIN,3915,18.6431,20.3490,19.4157,21.3272,30768.1364
6,VEDOLIZUMAB,OFF LABEL USE,3700,6.0103,10.0561,9.6217,10.5102,15730.8634
7,DUPIXENT,RASH,3664,4.0292,4.3017,4.1433,4.4661,6852.1526
8,DUPIXENT,INJECTION SITE PAIN,3555,4.9664,5.3116,5.1092,5.5220,8801.5276
9,DUPIXENT,ECZEMA,3193,42.3929,45.6000,42.6629,48.7392,36019.6058



These define the building and scoring logic as functions that can be used elsewhere.

In [ ]:
from pyspark.sql import functions as F


def build_drug_event_pairs(drug_df, reac_df):
    suspect_drugs = drug_df.filter(F.col("role_cod") == "PS") \
        .select("primaryid", "drugname")

    return suspect_drugs.join(
        reac_df.select("primaryid", "pt"),
        on="primaryid"
    ).select("primaryid", "drugname", "pt").distinct()


def compute_disproportionality(spark, drug_event_pairs):
    N = drug_event_pairs.select("primaryid").distinct().count()

    drug_event_counts = drug_event_pairs.groupBy("drugname", "pt") \
        .agg(F.countDistinct("primaryid").alias("a"))

    drug_totals = drug_event_pairs.groupBy("drugname") \
        .agg(F.countDistinct("primaryid").alias("drug_total"))

    event_totals = drug_event_pairs.groupBy("pt") \
        .agg(F.countDistinct("primaryid").alias("event_total"))

    contingency = (drug_event_counts
        .join(drug_totals, on="drugname")
        .join(event_totals, on="pt")
        .withColumn("b", F.col("drug_total") - F.col("a"))
        .withColumn("c", F.col("event_total") - F.col("a"))
        .withColumn("d", F.lit(N) - F.col("a") - F.col("b") - F.col("c"))
        .withColumn("N", F.lit(N))
    )

    for c in ["a", "b", "c", "d", "N"]:
        contingency = contingency.withColumn(c, F.col(c).cast("double"))

    # Continuity correction for zero cells
    contingency = (contingency
        .withColumn("_needs_correction",
            (F.col("a") == 0) | (F.col("b") == 0) |
            (F.col("c") == 0) | (F.col("d") == 0))
        .withColumn("a_adj",
            F.when(F.col("_needs_correction"), F.col("a") + 0.5).otherwise(F.col("a")))
        .withColumn("b_adj",
            F.when(F.col("_needs_correction"), F.col("b") + 0.5).otherwise(F.col("b")))
        .withColumn("c_adj",
            F.when(F.col("_needs_correction"), F.col("c") + 0.5).otherwise(F.col("c")))
        .withColumn("d_adj",
            F.when(F.col("_needs_correction"), F.col("d") + 0.5).otherwise(F.col("d")))
    )

    # PRR
    contingency = contingency.withColumn("PRR",
        (F.col("a_adj") / (F.col("a_adj") + F.col("b_adj"))) /
        (F.col("c_adj") / (F.col("c_adj") + F.col("d_adj")))
    )

    # ROR
    contingency = contingency.withColumn("ROR",
        (F.col("a_adj") * F.col("d_adj")) / (F.col("b_adj") * F.col("c_adj"))
    )

    # Chi square with Yates' correction
    contingency = contingency.withColumn("chi2",
        (F.col("N") *
         F.pow(
             F.abs(F.col("a") * F.col("d") - F.col("b") * F.col("c")) - F.col("N") / 2,
             2
         )) /
        ((F.col("a") + F.col("b")) *
         (F.col("c") + F.col("d")) *
         (F.col("a") + F.col("c")) *
         (F.col("b") + F.col("d")))
    )

    # 95% CI for ROR
    contingency = (contingency
        .withColumn("SE_ln_ROR",
            F.sqrt(
                1 / F.col("a_adj") + 1 / F.col("b_adj") +
                1 / F.col("c_adj") + 1 / F.col("d_adj")
            ))
        .withColumn("ROR_lower",
            F.exp(F.log(F.col("ROR")) - 1.96 * F.col("SE_ln_ROR")))
        .withColumn("ROR_upper",
            F.exp(F.log(F.col("ROR")) + 1.96 * F.col("SE_ln_ROR")))
    )

    # Evans' criteria
    contingency = contingency.withColumn("is_signal",
        (F.col("PRR") >= 2) & (F.col("chi2") >= 4) & (F.col("a") >= 3)
    )

    return contingency